# CSRNet Model Evaluation

This notebook contains a complete pipeline for evaluating the performance of the trained CSRNet model.

In [ ]:
import torch
import pandas as pd
import numpy as np
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
import sqlite3
# Import the model and dataset classes from your scripts
from csrnet_model import CSRNet
from csrnet_dataset import CSRNetDataset

In [ ]:
# --- Configuration ---
# This should be the main folder for your experiment
base_dir = '<DATASET_ROOT>/Exp_06_Vorversuche_YOLO/'
dataset_root = os.path.join(base_dir, 'dataset')
db_path = os.path.join(base_dir, 'stripGen_results.db')
# Path to the trained model weights
model_path = os.path.join(base_dir, 'runs', 'csrnet_train', 'best_model.pth')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# --- 1. Load Data ---
print(f"Loading data from {db_path}...")
sqlite_connection = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM results", sqlite_connection)
sqlite_connection.close()

# Make sure image paths are absolute
df['image_path'] = df['image_path'].apply(lambda p: os.path.join(base_dir, p))
all_image_paths = df['image_path'].tolist()

## 1. Load Trained Model

In [ ]:
# --- Load the Model ---
model = CSRNet().to(device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval() # Set the model to evaluation mode
    print(f"Model loaded successfully from {model_path}")
else:
    print(f"Error: Model weights not found at {model_path}")

## 2. Inference and Visualization on a Single Image

In [ ]:
# --- Single Image Inference ---

# Define transformations (must be the same as used in training)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Get a list of validation images
val_image_dir = os.path.join(dataset_root, 'images', 'val')
val_images = [f for f in os.listdir(val_image_dir) if f.endswith(('.png', '.jpg'))]

if val_images:
    def view_image(image_index):
        image_name = val_images[image_index]
        image_path = os.path.join(val_image_dir, image_name)
        
        # Load image
        img_pil = Image.open(image_path).convert('RGB')
        img_tensor = transform(img_pil).unsqueeze(0).to(device)

        # --- Run Inference ---
        with torch.no_grad():
            pred_density_map = model(img_tensor)
        
        # Get predicted count
        pred_count = pred_density_map.detach().cpu().sum().numpy()
        
        # --- Get Ground Truth ---
        points_path = os.path.join(dataset_root, 'points_annotations', 'val', os.path.splitext(image_name)[0] + '.csv')
        gt_count = 0
        if os.path.exists(points_path):
            gt_df = pd.read_csv(points_path)
            gt_count = len(gt_df)
            
        # --- Visualization ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
        
        # Original Image
        ax1.imshow(img_pil)
        ax1.set_title(f'Original Image: {image_name}\nGround Truth Count: {gt_count}')
        ax1.axis('off')
        
        # Predicted Density Map
        pred_density_map_np = pred_density_map.squeeze().cpu().numpy()
        im = ax2.imshow(pred_density_map_np, cmap='jet')
        ax2.set_title(f'Predicted Density Map\nPredicted Count: {pred_count:.2f}')
        ax2.axis('off')
        fig.colorbar(im, ax=ax2)
        
        plt.tight_layout()
        plt.show()

    interact(view_image, image_index=IntSlider(min=0, max=len(val_images)-1, step=1, value=0))

else:
    print("No validation images found.")

## 3. Quantitative Evaluation on Validation Set

In [ ]:
# --- Full Validation Set Evaluation ---

val_dataset = CSRNetDataset(root_dir=dataset_root, split='val', transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

results = []
epoch_mae = 0.0
epoch_mse = 0.0

model.eval()
with torch.no_grad():
    progress_bar = tqdm.tqdm(val_loader, desc='Evaluating on Validation Set')
    for i, (image, density_map) in enumerate(progress_bar):
        image = image.to(device)
        
        # Get prediction
        pred_density_map = model(image)
        
        # Get counts
        pred_count = pred_density_map.detach().cpu().sum().item()
        gt_count = density_map.detach().cpu().sum().item()
        
        # Store results
        results.append({'image_index': i, 'gt_count': gt_count, 'pred_count': pred_count})
        
        # Update metrics
        epoch_mae += abs(pred_count - gt_count)
        epoch_mse += (pred_count - gt_count)**2

# Calculate final metrics
final_mae = epoch_mae / len(val_dataset)
final_mse = np.sqrt(epoch_mse / len(val_dataset))

print(f"\n--- Evaluation Complete ---")
print(f"Mean Absolute Error (MAE): {final_mae:.2f}")
print(f"Mean Squared Error (MSE): {final_mse:.2f}")

results_df = pd.DataFrame(results)

## 4. Analysis of Results

In [ ]:
# --- Visualize Prediction vs. Ground Truth ---
if 'results_df' in locals():
    plt.figure(figsize=(10, 10))
    
    # Scatter plot of predictions vs ground truth
    sns.scatterplot(data=results_df, x='gt_count', y='pred_count', alpha=0.6)

    # Plot the y=x line for reference (perfect prediction)
    max_val = max(results_df['gt_count'].max(), results_df['pred_count'].max())
    plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Perfect Prediction')
    
    plt.title('Predicted Count vs. Ground Truth Count', fontsize=16)
    plt.xlabel('Ground Truth Count', fontsize=12)
    plt.ylabel('Predicted Count', fontsize=12)
    plt.grid(True)
    plt.legend()
    plt.axis('equal')
    plt.show()
else:
    print("Results DataFrame not found. Please run the evaluation cell first.")